# 17.4 时序差分学习 / Temporal-Difference (TD) Learning

**中文**：蒙特卡洛(17.3)必须**等一整局结束**、拿到真实回报才能更新——慢、方差大、还只能用于回合制任务。动态规划(17.2)每步就能更新，但**需要已知模型**。**时序差分(Temporal-Difference, TD)学习** 把两者的优点合二为一:像 MC 一样**无模型、从经验学**，又像 DP 一样**每走一步就更新(自举 bootstrap)**——不等回合结束，用"当前对下一状态的估计"来更新当前状态。TD 是现代 RL 的**真正基石**(Q-learning、SARSA、乃至 DQN 都是它的后裔)。
**English**: Monte Carlo (17.3) must **wait for a whole episode to end** and get the real return before updating — slow, high-variance, and episodic-only. Dynamic programming (17.2) updates every step but **needs a known model**. **Temporal-Difference (TD) learning** fuses the best of both: like MC it is **model-free, learning from experience**, and like DP it **updates every single step (bootstrapping)** — without waiting for the episode to end, using "the current estimate of the next state" to update the current one. TD is the **true cornerstone** of modern RL (Q-learning, SARSA, even DQN descend from it).

---

**中文**：**TD(0) 更新公式**——每走一步 $(s,a,r,s')$ 就更新:
**English**: The **TD(0) update** — after each step $(s,a,r,s')$:

$$V(s)\leftarrow V(s) + \alpha\underbrace{\big[\,r+\gamma V(s') - V(s)\,\big]}_{\text{TD 误差 } \delta}$$

**中文**：逐项拆解:
**English**: Breaking it down:
- **TD 目标 $r+\gamma V(s')$**:对回报的一步估计——即时奖励 + 折扣后的**下一状态当前估值**。这个"用估计更新估计"就是**自举(bootstrapping)**。
  **TD target $r+\gamma V(s')$**: a one-step estimate of the return — immediate reward + discounted **current estimate of the next state's value**. Using an estimate to update an estimate is **bootstrapping**.
- **TD 误差 $\delta=r+\gamma V(s')-V(s)$**:"实际看到的" 减 "原本预期的"——**惊喜程度**。$\alpha$ 是学习率,把估值往误差方向挪一点。
  **TD error $\delta=r+\gamma V(s')-V(s)$**: "what actually happened" minus "what was expected" — a measure of **surprise**. $\alpha$ is the learning rate, nudging the estimate toward the error.

**中文**：把 TD 用于**控制**(找最优策略),就得到 **SARSA**——它估计动作价值 $Q(s,a)$,更新用的是**实际采取的下一个动作 $a'$**:
**English**: Applying TD to **control** (finding the optimal policy) gives **SARSA** — it estimates action values $Q(s,a)$, updating with the **actually-taken next action $a'$**:

$$Q(s,a)\leftarrow Q(s,a)+\alpha\big[r+\gamma Q(s',a')-Q(s,a)\big]$$

**中文**：名字 **SARSA** 就来自更新用到的五元组 $(S,A,R,S',A')$。它是**同策略(on-policy)** 的——学习的是"**我实际在执行的(含探索的)ε-贪心策略**"的价值。
**English**: The name **SARSA** comes from the quintuple $(S,A,R,S',A')$ used in the update. It is **on-policy** — it learns the value of "**the ε-greedy policy I am actually following (exploration included)**."

> 💡 **面试速查 / Interview cheat-sheet（★★★ RL 核心必考）**
> **中文**：**TD=MC(采样,无模型) + DP(自举,每步更新)**。TD 目标=$r+\gamma V(s')$, TD 误差 $\delta$=惊喜。**vs MC**:TD **有偏但低方差**、可在线/每步更新、能处理连续不终止任务; MC 无偏但高方差、要等回合结束。**SARSA**=on-policy TD 控制(用实际 $a'$ 更新), 学"含探索的策略"的价值→行为更**保守/安全**。对比下节 **Q-learning**=off-policy(用 $\max_{a'}Q$ 更新), 学"贪心最优策略"→更激进。TD 误差 $\delta$ 还对应大脑**多巴胺**信号(神经科学彩蛋)。
> **English**: **TD = MC (sampling, model-free) + DP (bootstrapping, per-step updates)**. TD target = $r+\gamma V(s')$, TD error $\delta$ = surprise. **vs MC**: TD is **biased but low-variance**, updates online/every step, handles continuing (non-terminating) tasks; MC is unbiased but high-variance and must wait for episode end. **SARSA** = on-policy TD control (updates with the actual $a'$), learning the value of "the policy including exploration" → more **conservative/safe** behavior. Contrast next section's **Q-learning** = off-policy (updates with $\max_{a'}Q$), learning the "greedy optimal policy" → more aggressive. The TD error $\delta$ also mirrors the brain's **dopamine** signal (a neuroscience easter egg).


In [ ]:

# ============================================================
# 实验一:MC vs TD(0) 预测 —— 经典 1D 随机游走 / random walk (Sutton Ex 6.2)
# 中文:7 个状态排成一行(0..6), 两端(0,6)是终止态。从中间(3)出发, 每步等概率左/右。
#      走到最右端奖励+1, 其余奖励0。真实价值 V(1..5)=1/6,2/6,...,5/6(线性)。
#      这是对比 MC 与 TD 收敛性的教科书最小例子。
# English: 7 states in a line (0..6); ends (0,6) terminal. Start at 3, step left/right with equal prob.
#      Reaching the right end gives +1, else 0. True V(1..5)=1/6..5/6 (linear). The textbook minimal
#      example to compare MC vs TD convergence.
# ============================================================
import numpy as np, matplotlib.pyplot as plt
rng=np.random.default_rng(0)
true_V=np.array([0,1,2,3,4,5,0])/6                           # 真实价值(作为对比基准)/ ground-truth values
def walk_episode():
    s=3; traj=[]
    while s not in (0,6):
        ns=s+rng.choice([-1,1]); r=1.0 if ns==6 else 0.0     # 右端+1 / +1 at right end
        traj.append((s,r,ns)); s=ns
    return traj
def rms(V): return np.sqrt(np.mean((V[1:6]-true_V[1:6])**2)) # 与真值的均方根误差 / RMS error vs truth

def predict_TD(episodes, alpha):
    V=np.ones(7)*0.5; V[0]=V[6]=0; errs=[]
    for _ in range(episodes):
        for (s,r,ns) in walk_episode():
            V[s]+=alpha*(r + V[ns] - V[s])                    # TD(0):每步自举更新 / bootstrap per step
        errs.append(rms(V))
    return errs, V
def predict_MC(episodes, alpha):
    V=np.ones(7)*0.5; V[0]=V[6]=0; errs=[]
    for _ in range(episodes):
        traj=walk_episode(); G=traj[-1][1]                   # 回报=终局奖励(gamma=1)/ return = final reward
        for (s,r,ns) in traj:
            V[s]+=alpha*(G - V[s])                            # 常数α MC:回合末用真实回报更新 / MC update
        errs.append(rms(V))
    return errs, V

_,V_td=predict_TD(100,0.1); _,V_mc=predict_MC(100,0.1)
print("TD(0) 估计的 V(1..5) / TD estimate:", np.round(V_td[1:6],3))
print("真实值      true V(1..5):", np.round(true_V[1:6],3))
print("→ TD(0) 无需模型, 从经验就逼近了真实价值 / TD approximates true values from experience")


**中文**：做 **50 次独立重复**、对每个学习率对比 MC 与 TD(0) 的 RMS 误差随回合下降的曲线。这个实验揭示 TD 相对 MC 的核心优势——**方差更小、收敛更稳更快**。
**English**: Run **50 independent repetitions** and compare the RMS-error curves of MC vs TD(0) as episodes progress, per learning rate. This reveals TD's core advantage over MC — **lower variance, smoother and faster convergence**.


In [ ]:

# ============================================================
# MC vs TD(0) 收敛曲线(多次平均)/ MC vs TD convergence (averaged over runs)
# ============================================================
def avg_curve(fn, alpha, runs=50, episodes=100):
    return np.mean([fn(episodes,alpha)[0] for _ in range(runs)], axis=0)

fig,ax=plt.subplots(1,2,figsize=(13,4.6))
for alpha,c in zip([0.05,0.1,0.15],["#4C72B0","#55A868","#8172B3"]):
    ax[0].plot(avg_curve(predict_TD,alpha),color=c,label=f"TD α={alpha}")
for alpha,c in zip([0.05,0.1,0.15],["#C44E52","#DD8452","#937860"]):
    ax[0].plot(avg_curve(predict_MC,alpha),"--",color=c,label=f"MC α={alpha}")
ax[0].set_title("MC vs TD(0) 收敛:TD 误差更低更稳 / TD converges lower & smoother")
ax[0].set_xlabel("episode"); ax[0].set_ylabel("RMS error"); ax[0].legend(fontsize=8)
# 学到的价值 vs 真值 / learned values vs truth
ax[1].plot(range(1,6),true_V[1:6],"ko-",label="真实 true V")
ax[1].plot(range(1,6),V_td[1:6],"s-",color="#4C72B0",label="TD(0)")
ax[1].plot(range(1,6),V_mc[1:6],"^--",color="#C44E52",label="MC")
ax[1].set_title("100 回合后的价值估计 / value estimates after 100 eps"); ax[1].set_xlabel("state"); ax[1].set_ylabel("V"); ax[1].legend()
plt.tight_layout(); plt.savefig("/tmp/rl04_viz1.png",dpi=80); plt.show()
print("TD 的 RMS 误差全程低于 MC —— 低方差是 TD 的核心优势 / TD's low variance is its key edge")


**中文**：现在把 TD 用于**控制**——在经典的 **悬崖行走(Cliff Walking)** 环境上跑 **SARSA**。这个环境专门设计来展示 on-policy 的"性格":
**English**: Now apply TD to **control** — run **SARSA** on the classic **Cliff Walking** environment. This environment is designed to reveal on-policy "personality":

**中文**：4×12 网格，左下角出发、右下角是终点，二者之间的**整条底边是悬崖**。每走一步 −1；掉下悬崖 **−100** 并被送回起点。最短路是**贴着悬崖边走**(12 步)，但危险;稍微绕远一点走上面更安全。看 SARSA 学出哪条路。
**English**: A 4×12 grid: start bottom-left, goal bottom-right, and **the entire bottom edge between them is a cliff**. Each step is −1; falling off the cliff is **−100** and teleports back to the start. The shortest path **hugs the cliff edge** (12 steps) but is dangerous; going a bit higher is safer. Let's see which path SARSA learns.


In [ ]:

# ============================================================
# 实验二:悬崖行走 + SARSA / Cliff Walking + SARSA
# ============================================================
H,W=4,12; nS=H*W; nA=4; ACT=[(-1,0),(0,1),(1,0),(0,-1)]; ANM=["↑","→","↓","←"]
start=3*W+0; goal=3*W+11; cliff=set(3*W+c for c in range(1,11))
def cstep(s,a):
    r,c=divmod(s,W); dr,dc=ACT[a]
    nr,nc=max(0,min(H-1,r+dr)), max(0,min(W-1,c+dc)); ns=nr*W+nc
    if ns in cliff: return start,-100.0,False                # 掉崖:-100 回起点 / cliff: -100, back to start
    return ns, -1.0, (ns==goal)                              # 否则每步 -1 / else -1 per step

def sarsa(episodes=500, alpha=0.5, eps=0.1, gamma=1.0):
    Q=np.zeros((nS,nA)); ep_rewards=[]
    def act(s): return int(rng.integers(nA)) if rng.random()<eps else int(np.argmax(Q[s]))  # ε-贪心
    for _ in range(episodes):
        s=start; a=act(s); tot=0
        for _ in range(200):
            ns,r,done=cstep(s,a); tot+=r
            na=act(ns)                                        # 采样"实际下一个动作" a' / on-policy next action
            Q[s,a]+=alpha*(r + gamma*Q[ns,na]*(not done) - Q[s,a])   # SARSA 更新 / SARSA update
            s,a=ns,na
            if done: break
        ep_rewards.append(tot)
    return Q, ep_rewards

Q_sarsa, rewards = sarsa(500)
# 提取贪心路径 / extract greedy path
s=start; path=[s]
for _ in range(40):
    s=cstep(s,int(np.argmax(Q_sarsa[s])))[0]; path.append(s)
    if s==goal: break
print(f"SARSA 训练完, 最后50回合平均回报 / last-50 avg return: {np.mean(rewards[-50:]):.1f}")
print("学到的贪心路径长度 / greedy path length:", len(path)-1, "步")


In [ ]:

# ============================================================
# 可视化 SARSA / visualize SARSA
# ============================================================
fig,ax=plt.subplots(1,2,figsize=(14,4.4))
# ① 学习曲线(平滑)/ smoothed learning curve
rw=np.array(rewards); sm=np.convolve(rw,np.ones(20)/20,mode="valid")
ax[0].plot(rw,alpha=0.3,color="#4C72B0"); ax[0].plot(range(len(sm)),sm,color="#C44E52",lw=2,label="20-ep 平滑")
ax[0].set_title("SARSA 学习曲线:回报逐渐上升 / learning curve"); ax[0].set_xlabel("episode"); ax[0].set_ylabel("每回合总回报 return"); ax[0].set_ylim(-200,0); ax[0].legend()
# ② 学到的策略与路径 / learned policy + path
board=np.zeros((H,W))
for c in range(1,11): board[3,c]=-1                          # 悬崖 / cliff
ax[1].imshow(board,cmap="Reds_r",vmin=-1,vmax=1)
for s in range(nS):
    r,c=divmod(s,W)
    if s==goal: t="G"
    elif s==start: t="S"
    elif s in cliff: t="✗"
    else: t=ANM[int(np.argmax(Q_sarsa[s]))]
    ax[1].text(c,r,t,ha="center",va="center",fontsize=9)
pr=[divmod(s,W) for s in path]
ax[1].plot([c for r,c in pr],[r for r,c in pr],"o-",color="#4C72B0",lw=2,ms=4,label="贪心路径")
ax[1].set_title("SARSA 学到'安全路径'(远离悬崖)/ SARSA's SAFE path"); ax[1].set_xticks([]); ax[1].set_yticks([]); ax[1].legend()
plt.tight_layout(); plt.savefig("/tmp/rl04_viz2.png",dpi=80); plt.show()
print("SARSA 倾向绕开悬崖走安全路(因为它把'ε探索时可能掉崖'的风险算进了价值)")
print("SARSA prefers a safe detour (it factors the risk of falling during ε-exploration into its values)")


**中文**：诚实解读:
**English**: Honest takeaways:

**中文**：
1. **TD 的低方差优势**:在随机游走上，TD(0) 的 RMS 误差全程低于 MC(同样学习率下)。因为 MC 用的是**整条轨迹的真实回报**(把一路的随机性都吸进来了，方差大)，而 TD 只用**一步真实奖励 + 对后续的估计**(方差小，代价是引入了自举偏差)。这是 RL 里的**偏差-方差权衡**。
2. **TD 无需等回合结束**:每走一步就更新，天然支持在线学习和**连续不终止**的任务(MC 做不到)。
3. **SARSA 学的是"安全路"**:它绕开悬崖走上面。原因很深刻——SARSA 是**同策略**的，它学习的是"**我实际执行的 ε-贪心策略**"的价值,而这个策略有 10% 概率乱走。贴着悬崖边走时，一次探索性乱走就可能掉崖(−100)，所以 SARSA 认为"崖边"价值低，选择安全绕行。**它把探索的风险算进了决策**。
4. **伏笔**:下一节的 **Q-learning** 是**异策略**的——它学"**假设永远贪心(不探索)**的最优策略"价值，会大胆地贴着悬崖走最短路。这个 SARSA(安全) vs Q-learning(最优但冒险) 的对比,是 RL 最经典的教学案例,下节揭晓。

**English**:
1. **TD's low-variance advantage**: on the random walk, TD(0)'s RMS error stays below MC's (same learning rate). MC uses the **whole trajectory's real return** (absorbing all the path's randomness → high variance), while TD uses **one real reward + an estimate of the rest** (low variance, at the cost of a bootstrapping bias). This is RL's **bias-variance tradeoff**.
2. **TD needs no episode end**: it updates every step, naturally supporting online learning and **continuing (non-terminating)** tasks (MC cannot).
3. **SARSA learns the "safe path"**: it detours above the cliff. The reason is deep — SARSA is **on-policy**, learning the value of "**the ε-greedy policy I actually follow**," which has a 10% chance of a random move. Hugging the cliff, one exploratory misstep can fall off (−100), so SARSA deems the cliff edge low-value and detours. **It factors exploration risk into decisions.**
4. **Foreshadowing**: next section's **Q-learning** is **off-policy** — it learns the value of the "**always-greedy (no exploration)** optimal policy," boldly hugging the cliff for the shortest path. The SARSA (safe) vs Q-learning (optimal-but-risky) contrast is RL's most classic teaching case — revealed next.

> 💼 **实战视角 / Practical angle**
> **中文**:TD 是现代 RL 的**发动机**——几乎所有深度 RL(DQN、A2C、PPO 的价值网络)都用 TD 误差训练 critic。**TD 误差 δ 就是训练信号**:预测得越准 δ 越小。神经科学发现大脑的**多巴胺神经元**编码的正是类似 TD 误差的"奖励预测误差", RL 和生物学在此惊人相通。工程上:α(学习率)、λ(TD(λ) 的多步折中)、on-policy vs off-policy 是核心设计选择。面试金句:*"TD 用自举把 MC 的高方差换成低方差+小偏差, 每步在线更新; SARSA(on-policy)学含探索的策略更安全, Q-learning(off-policy)学贪心最优更激进——这就是悬崖行走里两条路的由来。"*
> **English**: TD is the **engine** of modern RL — nearly all deep RL (DQN, and the critics in A2C/PPO) trains via TD error. **The TD error δ is the training signal**: better predictions mean smaller δ. Neuroscience found the brain's **dopamine neurons** encode exactly a TD-like "reward prediction error" — RL and biology strikingly converge here. Engineering: α (learning rate), λ (the multi-step compromise of TD(λ)), and on- vs off-policy are core design choices. Interview line: *"TD's bootstrap trades MC's high variance for low variance + small bias, updating online every step; SARSA (on-policy) learns the exploration-inclusive policy (safer), Q-learning (off-policy) learns the greedy optimum (bolder) — the origin of the two cliff-walking paths."*

---
### 小结 / Summary
- **中文**:TD=MC(无模型采样)+DP(自举每步更新); TD 误差 δ=r+γV(s')−V(s) 是惊喜/训练信号。
- **English**: TD = MC (model-free sampling) + DP (bootstrapping per-step); TD error δ=r+γV(s')−V(s) is the surprise/training signal.
- **中文**:TD 比 MC 低方差、可在线、能处理连续任务(代价是自举偏差)。
- **English**: TD has lower variance than MC, works online, and handles continuing tasks (at the cost of bootstrapping bias).
- **中文**:SARSA=on-policy TD 控制(用实际 a' 更新), 学"含探索"的策略→安全路; 引出下节 off-policy 的 Q-learning。
- **English**: SARSA = on-policy TD control (updates with the actual a'), learning the exploration-inclusive policy → safe path; motivating off-policy Q-learning next.
